# Random Forest — Win64 Files
Malware detection using static analysis features from EMBER2024.  
File type: `Win64` executables  
Model: Random Forest

## 1. Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, log_loss, roc_auc_score, roc_curve,
    average_precision_score
)

## 2. Config

In [ ]:
# ---- CHANGE THIS ----
BASE_PATH = r"C:\Users\makis\ember_data\win64_clean"
# ---------------------

DIM          = 2568
RANDOM_STATE = 13

# EMBER2024 feature group offsets
GENERAL_START,  GENERAL_END  = 0,    7      # GeneralFileInfo       (7)
HIST_START,     HIST_END     = 7,    263    # ByteHistogram         (256)
ENTROPY_START,  ENTROPY_END  = 263,  519    # ByteEntropyHistogram  (256)
STRING_START,   STRING_END   = 519,  696    # StringExtractor       (177)
HEADER_START,   HEADER_END   = 696,  770    # HeaderFileInfo        (74)
SECTION_START,  SECTION_END  = 770,  994    # SectionInfo           (224)
IMPORT_START,   IMPORT_END   = 994,  2276   # Imports               (1282)
EXPORT_START,   EXPORT_END   = 2276, 2405   # Exports               (129)
DATADIR_START,  DATADIR_END  = 2405, 2439   # DataDirectories       (34)
RICH_START,     RICH_END     = 2439, 2472   # RichHeader            (33)
AUTH_START,     AUTH_END     = 2472, 2480   # AuthenticodeSignature (8)
PEWARN_START,   PEWARN_END   = 2480, 2568   # PEFormatWarnings      (88)
DOS_START,      DOS_END      = 753,  770    # DOS Header (inside HeaderFileInfo)

## 3. Load Data

In [ ]:
X_train_full = np.memmap(os.path.join(BASE_PATH, "X_train.dat"), dtype=np.float32, mode="r").reshape(-1, DIM)
y_train      = np.memmap(os.path.join(BASE_PATH, "y_train.dat"), dtype=np.int32,   mode="r")

X_test_full  = np.memmap(os.path.join(BASE_PATH, "X_test.dat"),  dtype=np.float32, mode="r").reshape(-1, DIM)
y_test       = np.memmap(os.path.join(BASE_PATH, "y_test.dat"),  dtype=np.int32,   mode="r")

print(f"Train: {X_train_full.shape[0]:,} samples | Test: {X_test_full.shape[0]:,} | Features: {DIM}")
print("Train labels:", dict(pd.Series(y_train).value_counts()))
print("Test  labels:", dict(pd.Series(y_test).value_counts()))

## 4. Helpers

In [ ]:
def evaluate(model, X, y, label=""):
    t0 = time.perf_counter()
    proba = model.predict_proba(X)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    inf_time = time.perf_counter() - t0

    acc  = accuracy_score(y, pred)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    ll   = log_loss(y, proba)
    roc  = roc_auc_score(y, proba)
    pr   = average_precision_score(y, proba)
    cm   = confusion_matrix(y, pred)

    print(f"\n=== {label} ===")
    print(f"Inference time : {inf_time:.4f} s")
    print(f"Accuracy       : {acc:.6f}")
    print(f"Precision      : {prec:.6f}")
    print(f"Recall         : {rec:.6f}")
    print(f"F1             : {f1:.6f}")
    print(f"Log Loss       : {ll:.6f}")
    print(f"ROC-AUC        : {roc:.6f}")
    print(f"PR-AUC         : {pr:.6f}")
    print(f"Confusion Matrix:\n{cm}")

    fpr, tpr, _ = roc_curve(y, proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"{label} (AUC={roc:.4f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {label}")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    return {"label": label, "accuracy": acc, "f1": f1, "roc_auc": roc, "pr_auc": pr}

def run_experiment(X_tr, X_te, y_tr, y_te, model, label):
    print(f"\nFeatures: {X_tr.shape[1]} cols")
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    print(f"Training time: {time.perf_counter() - t0:.2f} s")
    return evaluate(model, X_te, y_te, label)

## 5. Baseline Model

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample"
)

print("Training baseline Random Forest...")
t0 = time.perf_counter()
rf_baseline.fit(X_train_full, y_train)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(rf_baseline, X_test_full, y_test, "RF Baseline — Full Features")

## 6. Hyperparameter Tuning (RandomizedSearchCV on 50k subset)

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

SUBSET_SIZE = 50_000
N_ITER      = 50

half  = SUBSET_SIZE // 2
idx_0 = np.where(y_train == 0)[0]
idx_1 = np.where(y_train == 1)[0]

np.random.seed(RANDOM_STATE)
subset_idx = np.concatenate([np.random.choice(idx_0, half, replace=False),
                              np.random.choice(idx_1, half, replace=False)])
np.random.shuffle(subset_idx)

X_sub = X_train_full[subset_idx]
y_sub = y_train[subset_idx]

param_dist = {
    "n_estimators":      [100, 200, 300],
    "criterion":         ["gini", "entropy"],
    "max_depth":         [30, 45, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2, 5],
    "max_features":      ["sqrt", 0.05, 0.1],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE, bootstrap=True),
    param_distributions=param_dist,
    n_iter=N_ITER,
    scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc"},
    refit="pr_auc", cv=cv, verbose=2,
    random_state=RANDOM_STATE, n_jobs=1
)

print("Running RandomizedSearchCV...")
t0 = time.perf_counter()
search.fit(X_sub, y_sub)
print(f"Search time: {time.perf_counter() - t0:.2f} s")
print("Best PR-AUC:", search.best_score_)
print("Best params:", search.best_params_)

## 7. Tuned Model — Full Training & Evaluation

In [ ]:
# Best params found via RandomizedSearchCV
BEST_PARAMS = {
    "n_estimators":      100,
    "criterion":         "entropy",
    "max_features":      0.05,
    "max_depth":         45,
    "min_samples_split": 2,
    "min_samples_leaf":  1,
    "class_weight":      "balanced_subsample",
}

rf_tuned = RandomForestClassifier(
    **BEST_PARAMS, bootstrap=True, n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1
)

print("Training tuned Random Forest on full training set...")
t0 = time.perf_counter()
rf_tuned.fit(X_train_full, y_train)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(rf_tuned, X_test_full, y_test, "RF Tuned — Full Features")

## 8. Feature Group Experiments

In [ ]:
def get_rf():
    return RandomForestClassifier(**BEST_PARAMS, bootstrap=True,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=1)

def run_exp(X_tr, X_te, label):
    return run_experiment(X_tr, X_te, y_train, y_test, get_rf(), label)

In [ ]:
# ByteHistogram + ByteEntropyHistogram
run_exp(X_train_full[:, HIST_START:ENTROPY_END],
        X_test_full[:,  HIST_START:ENTROPY_END],
        "RF — ByteHistogram + ByteEntropyHistogram")

In [ ]:
# Imports only
run_exp(X_train_full[:, IMPORT_START:IMPORT_END],
        X_test_full[:,  IMPORT_START:IMPORT_END],
        "RF — Imports only")

In [ ]:
# Exports only
run_exp(X_train_full[:, EXPORT_START:EXPORT_END],
        X_test_full[:,  EXPORT_START:EXPORT_END],
        "RF — Exports only")

In [ ]:
# New EMBER2024 features
X_tr_new = np.hstack([X_train_full[:, DOS_START:DOS_END], X_train_full[:, DATADIR_START:PEWARN_END]])
X_te_new = np.hstack([X_test_full[:,  DOS_START:DOS_END], X_test_full[:,  DATADIR_START:PEWARN_END]])
run_exp(X_tr_new, X_te_new, "RF — New EMBER2024 Features")

In [ ]:
# General + Strings + Header + Section (best combo)
X_tr_c = np.hstack([X_train_full[:, GENERAL_START:GENERAL_END], X_train_full[:, STRING_START:SECTION_END]])
X_te_c = np.hstack([X_test_full[:,  GENERAL_START:GENERAL_END], X_test_full[:,  STRING_START:SECTION_END]])
run_exp(X_tr_c, X_te_c, "RF — General + Strings + Header + Section")

In [ ]:
# EMBER2018-equivalent features
X_tr_old = np.hstack([X_train_full[:, 0:DOS_START], X_train_full[:, DOS_END:DATADIR_START]])
X_te_old = np.hstack([X_test_full[:,  0:DOS_START], X_test_full[:,  DOS_END:DATADIR_START]])
run_exp(X_tr_old, X_te_old, "RF — EMBER2018-equivalent Features")